# DialogHelper Advanced Tests

This notebook tests additional dialoghelper functions not covered in `test_dialoghelper.ipynb`:

- **Multi-string replacement** - `msg_strs_replace()`
- **Line manipulation** - `msg_replace_lines()`, `msg_del_lines()`
- **Script injection** - `add_scr()`
- **Data transfer** - `pop_data()`, `fire_event()`
- **Advanced iife patterns** - Async operations, return values, error handling

**Prerequisites:** Run `test_dialoghelper.ipynb` first to verify basic functionality.

In [ ]:
# Cell 1: Setup & Import
from dialoghelper import dh_settings
dh_settings['port'] = 8000  # Dialeng server port

from dialoghelper import (
    read_msg, find_msgs, update_msg, add_msg, del_msg,
    msg_str_replace, msg_insert_line, msg_strs_replace, msg_replace_lines,
    msg_del_lines, add_html, add_scr, iife, fire_event, pop_data,
    curr_dialog, msg_idx
)
print("DialogHelper configured for advanced tests!")
print(f"Port: {dh_settings['port']}")

## Multi-String Replacement Tests

Test `msg_strs_replace()` which replaces multiple strings in a single call.

In [ ]:
# Cell 2: Test msg_strs_replace - Replace Multiple Strings
# Create a cell with multiple placeholders
test_cell_id = await add_msg(
    "Hello NAME! Welcome to PLACE. Today is DAY.",
    msg_type="note",
    placement="add_after"
)
print(f"Created cell: {test_cell_id}")

# Read original
original = await read_msg(id=test_cell_id)
print(f"Original: {original.content}")

# Replace multiple strings at once
await msg_strs_replace(
    id=test_cell_id,
    old_strs=["NAME", "PLACE", "DAY"],
    new_strs=["Alice", "Dialeng", "Monday"]
)

# Read back to verify
updated = await read_msg(id=test_cell_id)
print(f"After replace: {updated.content}")

In [ ]:
# Cell 3: Test msg_strs_replace on Code Cell
# Useful for refactoring variable names
code_cell_id = await add_msg(
    "old_var = 10\nresult = old_var * 2\nprint(old_var, result)",
    msg_type="code",
    placement="add_after"
)
print(f"Created code cell: {code_cell_id}")

# Read original
original = await read_msg(id=code_cell_id)
print(f"Original:\n{original.content}")

# Rename variable
await msg_strs_replace(
    id=code_cell_id,
    old_strs=["old_var"],
    new_strs=["new_var"]
)

# Read back
updated = await read_msg(id=code_cell_id)
print(f"\nAfter rename:\n{updated.content}")

## Line Manipulation Tests

Test `msg_replace_lines()` and `msg_del_lines()` for line-level editing.

In [ ]:
# Cell 4: Test msg_replace_lines - Replace Line Range
# Create a multi-line cell
test_cell_id = await add_msg(
    "Line 1: Header\nLine 2: Old content A\nLine 3: Old content B\nLine 4: Old content C\nLine 5: Footer",
    msg_type="note",
    placement="add_after"
)
print(f"Created cell: {test_cell_id}")

# Read original
original = await read_msg(id=test_cell_id)
print(f"Original:\n{original.content}")

# Replace lines 2-4 (1-based indexing) with new content
await msg_replace_lines(
    id=test_cell_id,
    start_line=2,
    end_line=4,
    new_content="Line 2: NEW content!\nLine 3: Also NEW!"
)

# Read back
updated = await read_msg(id=test_cell_id)
print(f"\nAfter replace_lines(2, 4):\n{updated.content}")

In [ ]:
# Cell 5: Test msg_del_lines - Delete Lines
# Create a cell with multiple lines
test_cell_id = await add_msg(
    "Keep this line\nDelete me 1\nDelete me 2\nKeep this too",
    msg_type="note",
    placement="add_after"
)
print(f"Created cell: {test_cell_id}")

original = await read_msg(id=test_cell_id)
print(f"Original:\n{original.content}")

# Delete lines 2-3 (1-based indexing)
await msg_del_lines(id=test_cell_id, start_line=2, end_line=3)

updated = await read_msg(id=test_cell_id)
print(f"\nAfter del_lines(2, 3):\n{updated.content}")

## Script Injection Tests

Test `add_scr()` which is a lower-level function for injecting scripts.

In [ ]:
# Cell 6: Test add_scr - Inject Script Element
# add_scr() is the underlying function used by iife()
await add_scr("""
    console.log('Script injected via add_scr!');
    
    const indicator = document.createElement('div');
    indicator.id = 'add-scr-test';
    indicator.style.cssText = `
        position: fixed;
        bottom: 20px;
        left: 20px;
        padding: 10px 20px;
        background: #2196F3;
        color: white;
        border-radius: 4px;
        z-index: 10000;
    `;
    indicator.textContent = 'add_scr() works!';
    document.body.appendChild(indicator);
    setTimeout(() => indicator.remove(), 3000);
""")
print("Script injected! Look at bottom-left corner.")

## Advanced iife() Patterns

More sophisticated JavaScript injection examples.

In [ ]:
# Cell 7: Test iife with Async Operations
await iife("""
    const delay = ms => new Promise(r => setTimeout(r, ms));
    
    console.log('Starting async operation...');
    
    const progress = document.createElement('div');
    progress.style.cssText = `
        position: fixed;
        top: 50%;
        left: 50%;
        transform: translate(-50%, -50%);
        padding: 30px;
        background: white;
        border-radius: 8px;
        box-shadow: 0 4px 20px rgba(0,0,0,0.3);
        z-index: 10000;
        font-size: 18px;
    `;
    document.body.appendChild(progress);
    
    for (let i = 1; i <= 5; i++) {
        progress.textContent = `Processing step ${i}/5...`;
        await delay(500);
    }
    
    progress.textContent = 'Done!';
    progress.style.background = '#4CAF50';
    progress.style.color = 'white';
    await delay(1000);
    progress.remove();
    
    console.log('Async operation complete!');
""")
print("Watch the center of the screen for progress!")

In [ ]:
# Cell 8: Test iife with Fetch API
await iife("""
    try {
        const response = await fetch(`/curr_dialog_`, {
            method: 'POST',
            headers: {'Content-Type': 'application/x-www-form-urlencoded'},
            body: `dlg_name=${window.NOTEBOOK_ID}`
        });
        const data = await response.json();
        
        console.log('Fetched dialog info:', data);
        
        const result = document.createElement('div');
        result.style.cssText = `
            position: fixed;
            top: 20px;
            left: 50%;
            transform: translateX(-50%);
            padding: 15px 25px;
            background: #1a1a2e;
            color: #16c79a;
            border-radius: 8px;
            font-family: monospace;
            z-index: 10000;
        `;
        result.innerHTML = `<strong>Notebook:</strong> ${data.name}<br><strong>Mode:</strong> ${data.mode}`;
        document.body.appendChild(result);
        setTimeout(() => result.remove(), 4000);
    } catch (err) {
        console.error('Fetch failed:', err);
    }
""")
print("Fetching notebook info from JS... check top of screen!")

In [ ]:
# Cell 9: Test iife - Query DOM and Log Cell Info
await iife("""
    const cells = document.querySelectorAll('.cell');
    const types = {};
    
    cells.forEach(cell => {
        const type = cell.dataset.type || 'unknown';
        types[type] = (types[type] || 0) + 1;
    });
    
    console.log('Cell counts:', types);
    
    const summary = document.createElement('div');
    summary.style.cssText = `
        position: fixed;
        bottom: 20px;
        right: 20px;
        padding: 15px;
        background: #f8f9fa;
        border: 1px solid #dee2e6;
        border-radius: 8px;
        z-index: 10000;
        font-family: system-ui;
    `;
    summary.innerHTML = '<strong>Cell Summary:</strong><br>' + 
        Object.entries(types)
            .map(([k, v]) => `${k}: ${v}`)
            .join('<br>');
    document.body.appendChild(summary);
    setTimeout(() => summary.remove(), 5000);
""")
print("Cell summary displayed in bottom-right corner!")

## Advanced Data Transfer Tests

Test `fire_event()` and `pop_data()` for bidirectional communication.

In [ ]:
# Cell 10: Setup Custom Event Handler
await iife("""
    if (window._mathHandler) {
        document.body.removeEventListener('do-math', window._mathHandler);
    }
    
    window._mathHandler = async (e) => {
        console.log('Math event received:', e.detail);
        const { operation, a, b, idx } = e.detail;
        let result;
        
        switch (operation) {
            case 'add': result = a + b; break;
            case 'multiply': result = a * b; break;
            case 'power': result = Math.pow(a, b); break;
            default: result = 'unknown operation';
        }
        
        await fetch('/push_data_blocking_', {
            method: 'POST',
            headers: {'Content-Type': 'application/x-www-form-urlencoded'},
            body: `dlg_name=${window.NOTEBOOK_ID}&data_id=${idx}&data=${JSON.stringify({result, operation, a, b})}`
        });
    };
    
    document.body.addEventListener('do-math', window._mathHandler);
    console.log('Math event handler registered!');
""")
print("Math event handler set up!")

In [ ]:
# Cell 11: Test fire_event + pop_data Pattern
import uuid

request_id = str(uuid.uuid4())[:8]

await fire_event('do-math', {
    'operation': 'power',
    'a': 2,
    'b': 10,
    'idx': request_id
})
print(f"Fired event with request_id: {request_id}")

try:
    response = await pop_data(idx=request_id, timeout=5)
    print(f"\nResult received!")
    print(f"  Operation: {response.operation}")
    print(f"  {response.a} ^ {response.b} = {response.result}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Cell 12: Test Multiple Event/Response Pairs
import uuid
import time

operations = [
    ('add', 100, 200),
    ('multiply', 7, 8),
    ('power', 3, 4)
]

print("Sending multiple calculations to browser...\n")

for op, a, b in operations:
    request_id = str(uuid.uuid4())[:8]
    
    await fire_event('do-math', {
        'operation': op,
        'a': a,
        'b': b,
        'idx': request_id
    })
    
    try:
        response = await pop_data(idx=request_id, timeout=5)
        print(f"{op}({a}, {b}) = {response.result}")
    except Exception as e:
        print(f"{op}({a}, {b}) failed: {e}")
    
    time.sleep(0.1)

print("\nAll calculations complete!")

## Utility Pattern Tests

Demonstrate useful patterns combining dialoghelper functions.

In [ ]:
# Cell 13: Utility - Duplicate Current Cell
# A pattern for duplicating the current cell's content

async def duplicate_cell(offset=0):
    """Duplicate a cell at the given offset (0=current, -1=previous, etc.)"""
    source_cell = await read_msg(n=offset)
    new_id = await add_msg(
        source_cell.content,
        msg_type=source_cell.type,
        placement="add_after"
    )
    return new_id

# Duplicate the cell 2 positions back (the utility section header)
new_id = await duplicate_cell(-2)
print(f"Duplicated cell to: {new_id}")
print("Check - a copy of the markdown header should appear below!")

In [ ]:
# Cell 14: Utility - Backup Cell Before Modification
# Create a backup of a cell, modify original, and restore if needed

async def backup_and_modify(cell_id, new_content):
    """Backup cell content, then modify it. Returns backup ID."""
    # Read original
    original = await read_msg(id=cell_id)
    
    # Create backup (hidden note)
    backup_id = await add_msg(
        f"# BACKUP of {cell_id}\n{original.content}",
        msg_type="note",
        placement="add_before"
    )
    
    # Modify original
    await update_msg(id=cell_id, content=new_content)
    
    return backup_id

# Create a test cell
test_id = await add_msg("Original important content", msg_type="note", placement="add_after")
print(f"Created test cell: {test_id}")

# Backup and modify
backup_id = await backup_and_modify(test_id, "Modified content (backup exists)")
print(f"Backup created: {backup_id}")

# Verify
modified = await read_msg(id=test_id)
backup = await read_msg(id=backup_id)
print(f"\nModified cell: {modified.content}")
print(f"Backup cell: {backup.content[:50]}...")

In [ ]:
# Cell 15: Utility - Find and Replace Across All Cells
# Search and replace a string in all matching cells

async def find_replace_all(pattern, old_str, new_str, msg_type=None):
    """Find cells matching pattern and replace string in all of them."""
    # Find matching cells
    cells = await find_msgs(re_pattern=pattern, msg_type=msg_type, limit=100)
    count = 0
    
    for cell in cells:
        if old_str in cell.content:
            await msg_str_replace(id=cell.id, old_str=old_str, new_str=new_str)
            count += 1
    
    return count

# Create some test cells with a common pattern
for i in range(3):
    await add_msg(f"Test cell {i}: Replace TARGET here", msg_type="note", placement="add_after")

print("Created 3 test cells with 'TARGET'")

# Replace in all matching cells
replaced = await find_replace_all("TARGET", "TARGET", "REPLACED")
print(f"\nReplaced 'TARGET' with 'REPLACED' in {replaced} cells")
print("Check the cells below to verify!")

## Cleanup

Optional: Clean up test cells created during this notebook.

In [ ]:
# Cell 16: Cleanup Test Cells (OPTIONAL)
# Run this to remove cells created by tests

# Find cells created by tests (they have certain patterns)
patterns_to_clean = [
    "Auto-generated",
    "BACKUP of",
    "Test cell",
    "REPLACED",
    "Original important",
    "Modified content"
]

deleted_count = 0
for pattern in patterns_to_clean:
    cells = await find_msgs(re_pattern=pattern, limit=50)
    for cell in cells:
        try:
            await del_msg(id=cell.id)
            deleted_count += 1
        except:
            pass

print(f"Cleaned up {deleted_count} test cells")

## Summary

This notebook tested additional dialoghelper functions:

### Multi-String Operations
- **msg_strs_replace()** - Replace multiple strings at once
- **msg_replace_lines()** - Replace a range of lines

### Script Injection
- **add_scr()** - Lower-level script injection
- **Advanced iife()** - Async patterns, fetch API, DOM queries

### Data Transfer
- **fire_event()** - Fire custom events to browser
- **pop_data()** - Receive data from browser

### Utility Patterns
- Cell duplication
- Backup before modify
- Find/replace across cells

These patterns enable powerful notebook automation and customization!